In [ ]:
#!/usr/bin/env python3
"""
Run Claude Sonnet 4.6 across the full 4-student x 2-day x 5-behavior evaluation
matrix and score its output against the human-validated gold standard.

Configuration (temperature, max_tokens, prompts, retry policy) matches the
GPT-4o and GPT-5.2 runs so that results are directly comparable across models.

Serial execution only. Idempotent: runs that already completed successfully
(return_code == 0) are skipped automatically; failed runs are retried.

Usage:
  python3 evaluate_model.py                      run all 40 combinations
  python3 evaluate_model.py --only "Taylor Swift" "day 1" "planning"
  python3 evaluate_model.py --status              print completion status only
"""
import argparse
import csv
import json
import os
import re
import statistics
import time
import traceback
from contextlib import redirect_stdout, redirect_stderr
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import requests

## Configuration

In [ ]:
PROJECT_ROOT  = Path("/Users/caiwansun/Desktop/LLM_L_AND_I_PROJECT")
DATA_DIR      = PROJECT_ROOT / "study-data-per-student-day-behavior"
PROMPTS_DIR   = PROJECT_ROOT / "all-prompts"
RESULTS_ROOT  = PROJECT_ROOT / "results"
FIGURES_ROOT  = PROJECT_ROOT / "figures"
GOLD_CSV_PATH = Path("/Users/caiwansun/Downloads/gold-human-validation - gold.csv")
REF_DIR       = PROJECT_ROOT / "original GPT 4o results"

STUDENTS  = ["Taylor Swift", "DaPaw", "Rose", "SJ3747"]
DAYS      = ["day 1", "day 2"]
BEHAVIORS = ["enacting", "planning", "reflecting", "monitoring", "interacting"]
MODEL_ID, MODEL_TAG = "us.anthropic.claude-sonnet-4-6", "sonnet4_6"

# All three evaluated models. Used to scope which model outputs are included
# when the (local, no-API-cost) gold-standard comparison is (re)computed for a
# given combination, so the comparison summary always covers all three models.
ALL_MODELS = [
    ("us.anthropic.claude-sonnet-4-6", "sonnet4_6"),
    ("gpt-5.2", "gpt5_2"),
    ("gpt-4o", "gpt4o"),
]

N_RUNS             = 100
MAX_TOKENS         = 8000
MAX_ATTEMPTS       = 3
BASE_URL           = "https://prod-api.vanderbilt.ai"
SPLIT_STRING       = "\n[***NEW_MESSAGE***]\n"
REQUEST_TIMEOUT_S  = 900
ZERO_DUR_NORM_S    = 1.0

STUDENT_SLUGS = {"Taylor Swift": "taylor", "DaPaw": "dapaw", "Rose": "rose", "SJ3747": "sj3747"}
DAY_TAGS = {"day 1": "day1", "day 2": "day2"}
DAY_RAW  = {"day 1": "d1",   "day 2": "d2"}

# Taylor Swift / day 1 / enacting already has 100 completed runs at a legacy
# path for sonnet4_6 and gpt5_2; those are reused rather than re-run.
SKIP_RUNNING = {
    ("Taylor Swift", "day 1", "enacting", "sonnet4_6"),
    ("Taylor Swift", "day 1", "enacting", "gpt5_2"),
}
EXCLUDE_ALL = set()

## Utility functions

In [ ]:
def _load_amp_token():
    tok = os.environ.get("AMP_TOKEN", "").strip()
    if tok:
        return tok
    raise RuntimeError("AMP_TOKEN not found. Set env var AMP_TOKEN=<your-token>.")


AMP_TOKEN = _load_amp_token()
HDRS = {"Authorization": f"Bearer {AMP_TOKEN}", "Content-Type": "application/json"}


def get_runs_dir(stu_slug, day_raw, behavior, model_tag):
    day_tag = "day1" if day_raw == "d1" else "day2"
    legacy = RESULTS_ROOT / stu_slug / behavior / f"runs_{model_tag}"
    if stu_slug == "taylor" and day_raw == "d1" and behavior == "enacting" and legacy.exists():
        return legacy
    return RESULTS_ROOT / stu_slug / day_tag / behavior / f"runs_{model_tag}"


def parse_t(v):
    """Parse an 'H:MM:SS' or 'MM:SS' timestamp into seconds."""
    s = str(v or "").strip()
    if not s:
        return None
    p = s.split(":")
    try:
        if len(p) == 3:
            return int(p[0]) * 3600 + int(p[1]) * 60 + float(p[2])
        if len(p) == 2:
            return int(p[0]) * 60 + float(p[1])
    except ValueError:
        pass
    return None


def correct_interval(a, b, global_end_s):
    """Clamp an interval to [0, global_end_s] and enforce a minimum duration."""
    if b <= a:
        b = a + ZERO_DUR_NORM_S
    a = max(0.0, a)
    b = min(b, global_end_s)
    if b <= a:
        a = max(0.0, global_end_s - ZERO_DUR_NORM_S)
        b = global_end_s
    return a, b


def enrich_seg(seg, global_end_s):
    a = parse_t(seg.get("time_in", ""))
    b = parse_t(seg.get("time_out", ""))
    if a is None or b is None:
        return seg
    ac, bc = correct_interval(a, b, global_end_s)
    out = dict(seg)
    out["corrected_start_seconds"] = round(ac, 4)
    out["corrected_end_seconds"] = round(bc, 4)
    return out


def load_source_csv(student, day, behavior):
    """Load and filter the per-student session CSV for one behavior category."""
    path = DATA_DIR / f"L&I - embodied - Student {student} - {day} - {behavior}.csv"
    df = pd.read_csv(path).fillna("")
    ends = [parse_t(v) for v in df.get("end_time", [])]
    global_end_s = max(v for v in ends if v is not None)

    beh = behavior.lower()
    d = df.copy()
    if "data" in d.columns:
        d["data"] = d["data"].replace("not moving", "stationary")

    if beh in {"enacting", "monitoring"}:
        d = d[~d["modality"].isin(["gesture", "speech"])]
        d = d[(d["modality"] != "gaze") | (d["data"] == "Screen")]
    elif beh == "interacting":
        d = d[~d["modality"].isin(["movement", "action", "gesture"])]
    elif beh in {"reflecting", "planning"}:
        d = d[~d["modality"].isin(["movement", "action", "state"])]

    body = d.to_csv(index=False)
    if beh in {"interacting", "reflecting", "planning"}:
        body = f"Speaker: {student}\n\n" + body
    return body, global_end_s


_FORMAT_OVERRIDE = """

IMPORTANT — OUTPUT FORMAT (takes precedence over any prior instruction):
Return a single JSON object with key "segments" containing an array.
Each element is ONE action event — do NOT merge separate events even if they overlap in time.

Required schema for every element:
{
  "time_in":  "H:MM:SS or MM:SS",
  "time_out": "H:MM:SS or MM:SS",
  "label":    "L" or "I",
  "action":   "brief description, one phrase or sentence"
}

Rules:
- One JSON object per distinct action event
- label must be exactly "L" or "I" (uppercase single letter)
- No markdown fences, no prose outside the JSON
- No extra keys beyond time_in, time_out, label, action
"""


def load_prompt(behavior, model_id):
    """Build the few-shot message list for one behavior. Claude models receive
    user/user/assistant roles; other models receive system/user/assistant."""
    path = PROMPTS_DIR / f"L_and_I_Prompt_{behavior.upper()}.txt"
    raw = path.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")
    parts = raw.split(SPLIT_STRING)
    if len(parts) != 3:
        raise RuntimeError(f"{path.name}: expected 3 blocks, got {len(parts)}")
    is_claude = "claude" in model_id.lower() or "anthropic" in model_id.lower()
    if is_claude:
        return [
            {"role": "user", "content": parts[0].strip() + _FORMAT_OVERRIDE},
            {"role": "user", "content": parts[1].strip()},
            {"role": "assistant", "content": parts[2].strip()},
        ]
    return [
        {"role": "system", "content": parts[0].strip() + _FORMAT_OVERRIDE},
        {"role": "user", "content": parts[1].strip()},
        {"role": "assistant", "content": parts[2].strip()},
    ]


def compute_metrics(segments, global_end_s):
    """Compute segment count, raw total duration, and de-overlapped coverage."""
    intervals, raw_s = [], 0.0
    for seg in (segments if isinstance(segments, list) else []):
        if not isinstance(seg, dict):
            continue
        a = seg.get("corrected_start_seconds")
        b = seg.get("corrected_end_seconds")
        if a is None or b is None:
            a0 = parse_t(seg.get("time_in", ""))
            b0 = parse_t(seg.get("time_out", ""))
            if a0 is None or b0 is None:
                continue
            a, b = correct_interval(a0, b0, global_end_s)
        raw_s += b - a
        intervals.append((a, b))
    intervals.sort()
    merged = []
    for ia, ib in intervals:
        if merged and ia <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], ib))
        else:
            merged.append([ia, ib])
    cov_s = sum(ib - ia for ia, ib in merged)
    return {
        "n_segments": len(segments) if isinstance(segments, list) else 0,
        "total_duration_seconds": round(raw_s, 4),
        "total_duration_minutes": round(raw_s / 60.0, 4),
        "coverage_duration_seconds": round(cov_s, 4),
        "coverage_duration_minutes": round(cov_s / 60.0, 4),
    }


def _unwrap_fence(text):
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def _salvage_segs(s):
    """Recover as many well-formed {time_in, time_out, ...} objects as possible
    from a string that may not be valid JSON overall (e.g. a truncated response)."""
    segs, i, n = [], 0, len(s)
    while i < n:
        oi = s.find("{", i)
        if oi < 0:
            break
        depth, in_str, esc = 0, False, False
        j = oi
        while j < n:
            c = s[j]
            if esc:
                esc = False
            elif in_str:
                if c == "\\":
                    esc = True
                elif c == '"':
                    in_str = False
            else:
                if c == '"':
                    in_str = True
                elif c == "{":
                    depth += 1
                elif c == "}":
                    depth -= 1
                    if depth == 0:
                        break
            j += 1
        if depth != 0:
            i = oi + 1
            continue
        try:
            obj = json.loads(s[oi:j + 1])
            if isinstance(obj, dict) and "time_in" in obj and "time_out" in obj:
                segs.append(obj)
        except Exception:
            pass
        i = j + 1
    return segs


def robust_parse(raw_text):
    """Parse a model response into {"segments": [...]}, trying progressively
    more permissive strategies and returning a status string for each."""
    if not raw_text:
        return None, "empty"
    s = _unwrap_fence(raw_text)

    def _norm(obj):
        if isinstance(obj, dict) and "segments" in obj:
            return {"segments": [x for x in obj["segments"] if isinstance(x, dict)]}
        if isinstance(obj, list):
            return {"segments": [x for x in obj if isinstance(x, dict)]}
        if isinstance(obj, dict) and "time_in" in obj and "time_out" in obj:
            return {"segments": [obj]}
        return None

    try:
        r = _norm(json.loads(s))
        if r is not None:
            return r, "clean"
    except Exception:
        pass
    try:
        obj, _ = json.JSONDecoder().raw_decode(s)
        r = _norm(obj)
        if r is not None:
            return r, "raw_decode"
    except Exception:
        pass
    for pat in (r"\{[\s\S]*\}", r"\[[\s\S]*\]"):
        mm = re.search(pat, s)
        if not mm:
            continue
        try:
            r = _norm(json.loads(mm.group(0)))
            if r is not None:
                return r, "regex"
        except Exception:
            pass
    segs = _salvage_segs(s)
    if segs:
        return {"segments": segs}, "salvaged"
    return None, "failed"


def plot_series(xs, ys, ylabel, title, out_path, color="C0"):
    vals = [v for v in ys if v is not None]
    med = statistics.median(vals) if vals else None
    mean = statistics.mean(vals) if vals else None
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(xs, ys, marker="o", linewidth=1, color=color)
    if med is not None:
        ax.axhline(med, color="C2", linestyle="--", linewidth=1, label=f"median={med:.2f}")
    if mean is not None:
        ax.axhline(mean, color="C3", linestyle=":", linewidth=1, label=f"mean={mean:.2f}")
    if any(k in ylabel.lower() for k in ("count", "segment", "n_seg")):
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title(title)
    ax.set_xlabel("run")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

## Gold-standard table construction

In [ ]:
_STU_MAP  = {"dapaw": "DaPaw", "rose": "Rose", "sj3747": "SJ3747", "taylor": "Taylor_Swift"}
_DAY_MAP2 = {"d1": "Day1", "d2": "Day2"}
_SLUG_MAP = {"dapaw": "dapaw", "rose": "rose", "sj3747": "sj3747", "taylor": "taylor"}


def _load_ref_segs(label, day_std, stu_std):
    stem1 = f"{label}_{day_std}_{stu_std}".lower()
    stem2 = f"{label}_{day_std}_{stu_std.replace('_', ' ')}".lower()
    for f in REF_DIR.glob("*.txt"):
        if f.stem.lower() in (stem1, stem2):
            try:
                return json.loads(f.read_text()).get("segments", []), f.name
            except Exception:
                return [], f.name
    return None, f"{label}_{day_std}_{stu_std}.txt [NOT FOUND]"


_START_MATCH_TOL_S = 1e-6


def _resolve_gold_interval(segs, t):
    """Resolve a human-annotated timestamp t into a full gold interval by
    finding the GPT-4o reference segment(s) whose start time exactly equals t;
    ties are broken by earliest end time. Returns None if no reference segment
    starts exactly at t, so the caller can record it as unmatched."""
    exact_start = []
    for s in segs:
        ti = parse_t(s.get("time_in"))
        to = parse_t(s.get("time_out"))
        if ti is None or to is None:
            continue
        s0, s1 = min(ti, to), max(ti, to)
        if abs(s0 - t) <= _START_MATCH_TOL_S:
            exact_start.append((s1, s0, s1))
    if not exact_start:
        return None
    exact_start.sort()  # earliest end time first
    _, gs, ge = exact_start[0]
    if ge <= gs:
        ge = gs + ZERO_DUR_NORM_S
    return gs, ge


def _build_gold_table():
    gold_table = []
    unmatched = []
    with open(GOLD_CSV_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            label = row["label"].strip()
            seg = row["segment"].strip()
            corr = row["ai_correct"].strip().upper() == "TRUE"
            m = re.match(r'(d\d)\s*-\s*(\w+)\s*-\s*at\s+([\d:]+)', seg)
            if not m:
                unmatched.append({"segment": seg, "reason": "PARSE_FAIL"})
                continue
            day_raw, stu_raw, tstr = m.group(1), m.group(2).lower(), m.group(3)
            t = parse_t(tstr)
            if t is None:
                unmatched.append({"segment": seg, "reason": "TIME_PARSE_FAIL"})
                continue
            stu_std = _STU_MAP.get(stu_raw)
            stu_slug = _SLUG_MAP.get(stu_raw)
            if not stu_std:
                unmatched.append({"segment": seg, "reason": f"UNKNOWN_STUDENT:{stu_raw}"})
                continue
            day_std = _DAY_MAP2[day_raw]
            ref_segs, ref_file = _load_ref_segs(label, day_std, stu_std)
            if ref_segs is None:
                unmatched.append({"segment": seg, "reason": f"REF_MISSING:{ref_file}"})
                continue
            resolved = _resolve_gold_interval(ref_segs, t)
            if resolved is None:
                unmatched.append({"segment": seg, "reason": f"NO_EXACT_START_MATCH (t={t}s, ref={ref_file})"})
                continue
            gs, ge = resolved
            gold_table.append({
                "label": label, "stu_slug": stu_slug, "stu_std": stu_std,
                "day_raw": day_raw, "day_std": day_std,
                "gs": gs, "ge": ge, "ai_correct": corr, "segment": seg,
                "ref_file": ref_file,
            })
    if unmatched:
        print(f"  [gold-table] {len(unmatched)} unmatched entries: {unmatched}")
    return gold_table


GOLD_TABLE = _build_gold_table()
print(f"Gold table: {len(GOLD_TABLE)} entries built from {GOLD_CSV_PATH.name}")

## Gold-standard comparison

In [ ]:
TOL = 1e-4

CAT_NAMES = {
    1: "1  — Exact Equal               [SA=SG, EA=EG]",
    2: "2  — Start Before, End Equal   [SA<SG, EA=EG]",
    3: "3  — Start Equal, End After    [SA=SG, EA>EG]",
    4: "4  — Start Before, End Inside  [SA<SG, SG<EA<EG]",
    5: "5  — Start Inside, End After   [SG<SA<EG, EA>EG]",
    6: "6  — Start Equal, End Before   [SA=SG, EA<EG]",
    7: "7  — Start After, End Equal    [SG<SA<EG, EA=EG]",
    8: "8  — AI Fully Inside Gold      [SG<SA, EA<EG]",
    9: "9  — AI Contains Gold          [SA<SG, EA>EG]",
    10: "10 — Complete Miss / No Overlap [EA<=SG or SA>=EG]",
}


def classify_cat(SA, EA, SG, EG):
    """Classify a predicted interval [SA, EA] against a gold interval
    [SG, EG] into one of 10 boundary-relationship categories."""
    def eq(a, b):
        return abs(a - b) <= TOL

    overlap = max(0.0, min(EA, EG) - max(SA, SG))
    if overlap <= 0:
        return 10
    if eq(SA, SG) and eq(EA, EG):
        return 1
    if SA < SG - TOL and eq(EA, EG):
        return 2
    if eq(SA, SG) and EA > EG + TOL:
        return 3
    if SA < SG - TOL and SG + TOL < EA < EG - TOL:
        return 4
    if SG + TOL < SA < EG - TOL and EA > EG + TOL:
        return 5
    if eq(SA, SG) and EA < EG - TOL:
        return 6
    if SG + TOL < SA < EG - TOL and eq(EA, EG):
        return 7
    if SA > SG - TOL and EA < EG + TOL:
        return 8
    if SA < SG + TOL and EA > EG - TOL:
        return 9
    return 10


def find_best_match(segs, SG, EG):
    """Select the predicted segment that best matches a gold interval, by
    largest overlap, then highest IoU, then smallest boundary error."""
    best = None
    best_overlap, best_iou, best_berr = 0.0, -1.0, float("inf")
    for seg in (segs if isinstance(segs, list) else []):
        if not isinstance(seg, dict):
            continue
        SA = seg.get("corrected_start_seconds") or parse_t(seg.get("time_in", ""))
        EA = seg.get("corrected_end_seconds") or parse_t(seg.get("time_out", ""))
        if SA is None or EA is None:
            continue
        if EA <= SA:
            EA = SA + ZERO_DUR_NORM_S
        overlap = max(0.0, min(EA, EG) - max(SA, SG))
        if overlap <= 0:
            continue
        union = max(EA, EG) - min(SA, SG)
        iou = overlap / union if union > 0 else 0.0
        berr = abs(SA - SG) + abs(EA - EG)
        if (overlap > best_overlap
                or (overlap == best_overlap and iou > best_iou)
                or (overlap == best_overlap and iou == best_iou and berr < best_berr)):
            best_overlap, best_iou, best_berr = overlap, iou, berr
            best = (SA, EA)
    return best


def compare_gold_for_combo(student, day, behavior, model_tags, n_runs=None):
    """Score every run of each requested model against the gold entries for
    one (student, day, behavior) combination and write per-model CSVs plus a
    combined human-readable summary."""
    if n_runs is None:
        n_runs = N_RUNS
    beh = behavior.lower()
    stu_slug = STUDENT_SLUGS[student]
    day_raw = DAY_RAW[day]
    day_tag = DAY_TAGS[day]
    out_dir = RESULTS_ROOT / stu_slug / day_tag / beh
    out_dir.mkdir(parents=True, exist_ok=True)

    gold_entries = [g for g in GOLD_TABLE
                    if g["label"] == behavior.upper()
                    and g["stu_slug"] == stu_slug
                    and g["day_raw"] == day_raw]
    if not gold_entries:
        print(f"  [gold] No entries for {student} {day} {behavior.upper()}")
        return {}

    print(f"\n  GOLD COMPARISON: {student}  {day}  {behavior.upper()}")
    print(f"  {len(gold_entries)} gold entries x {len(model_tags)} models x {n_runs} runs")

    _fields = ["segment", "label", "gold_start", "gold_end", "ai_correct",
               "model_tag", "run", "category", "category_name", "SA", "EA", "overlap"]
    results = {}

    for model_tag in model_tags:
        runs_dir = get_runs_dir(stu_slug, day_raw, beh, model_tag)
        if not runs_dir.exists():
            print(f"  [{model_tag}] runs_dir not found: {runs_dir} — skipping")
            continue

        cat_all = {c: 0 for c in range(1, 11)}
        cat_true = {c: 0 for c in range(1, 11)}
        cat_false = {c: 0 for c in range(1, 11)}
        rows = []

        for gold in gold_entries:
            SG, EG, corr = gold["gs"], gold["ge"], gold["ai_correct"]
            for ri in range(1, n_runs + 1):
                sf = runs_dir / f"run{ri}" / "segments.json"
                if not sf.exists():
                    continue
                try:
                    segs = json.loads(sf.read_text()).get("segments", [])
                except Exception:
                    continue

                best = find_best_match(segs, SG, EG)
                if best is None:
                    cat, SA, EA, ov = 10, None, None, 0.0
                else:
                    SA, EA = best
                    cat = classify_cat(SA, EA, SG, EG)
                    ov = max(0.0, min(EA, EG) - max(SA, SG))

                cat_all[cat] += 1
                (cat_true if corr else cat_false)[cat] += 1
                rows.append({
                    "segment": gold["segment"], "label": gold["label"],
                    "gold_start": round(SG, 3), "gold_end": round(EG, 3),
                    "ai_correct": corr, "model_tag": model_tag, "run": ri,
                    "category": cat, "category_name": CAT_NAMES[cat],
                    "SA": round(SA, 3) if SA is not None else "",
                    "EA": round(EA, 3) if EA is not None else "",
                    "overlap": round(ov, 3),
                })

        csv_path = out_dir / f"gold_comparison_{model_tag}.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=_fields, extrasaction="ignore")
            w.writeheader()
            w.writerows(rows)
        print(f"  [{model_tag}] {len(rows)} comparisons -> {csv_path.name}")
        results[model_tag] = {"combined": cat_all, "correct": cat_true, "incorrect": cat_false}

    _print_gold_summary(results, gold_entries, out_dir, student, day, behavior)
    return results


def _print_gold_summary(results, gold_entries, out_dir, student, day, behavior):
    n_t = sum(1 for g in gold_entries if g["ai_correct"])
    n_f = len(gold_entries) - n_t
    lines = [
        f"\nGOLD SUMMARY: {student} | {day} | {behavior.upper()}",
        f"Gold entries: {len(gold_entries)}  (ai_correct TRUE={n_t}, FALSE={n_f})",
    ]
    for model_tag, res in results.items():
        total = sum(res["combined"].values())
        lines += [
            f"\n  -- {model_tag}  (total comparisons = {total}) --",
            f"  {'Cat':>3}  {'Description':<44}  {'All':>6}  {'TRUE':>6}  {'FALSE':>6}  {'%All':>6}",
            f"  {'---':>3}  {'-' * 44}  {'---':>6}  {'----':>6}  {'-----':>6}  {'----':>6}",
        ]
        for c in range(1, 11):
            pct = res["combined"][c] / total * 100 if total else 0
            lines.append(
                f"  {c:>3}  {CAT_NAMES[c]:<44}  "
                f"{res['combined'][c]:>6}  "
                f"{res['correct'][c]:>6}  "
                f"{res['incorrect'][c]:>6}  "
                f"{pct:>5.1f}%"
            )
    txt = "\n".join(lines)
    print(txt)
    sp = out_dir / "gold_comparison_summary.txt"
    sp.write_text(txt, encoding="utf-8")
    print(f"\n  Summary -> {sp}")

## Single model call

In [ ]:
def call_model_once(messages, model_id, run_dir, stem, global_end_s):
    is_claude = "claude" in model_id.lower() or "anthropic" in model_id.lower()
    payload = {"data": {
        "temperature": 0, "max_tokens": MAX_TOKENS,
        "dataSources": [], "messages": messages,
        "options": {"skipRag": True, "ragOnly": False, "model": {"id": model_id}},
    }}
    if not is_claude:
        payload["data"]["response_format"] = {"type": "json_object"}

    last_err = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            r = requests.post(f"{BASE_URL}/chat", headers=HDRS, json=payload, timeout=REQUEST_TIMEOUT_S)
        except Exception as e:
            last_err = e
            time.sleep(5 * attempt)
            continue

        (run_dir / f"raw_response_attempt{attempt}.txt").write_text(r.text[:50000], encoding="utf-8")
        if r.status_code >= 400:
            last_err = RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
            time.sleep(5 * attempt)
            continue

        result = r.json()
        if not result.get("success"):
            last_err = RuntimeError(f"API failure: {str(result)[:200]}")
            time.sleep(5 * attempt)
            continue

        raw_data = result.get("data", "")
        raw_text = (json.dumps(raw_data) if isinstance(raw_data, (dict, list)) else str(raw_data).strip())
        if not raw_text:
            raw_text = str(result.get("message", "")).strip()
        (run_dir / f"raw_response_attempt{attempt}.txt").write_text(raw_text[:50000], encoding="utf-8")

        parsed, status = robust_parse(raw_text)
        if parsed is not None:
            segs = [enrich_seg(s, global_end_s) for s in parsed.get("segments", []) if isinstance(s, dict)]
            (run_dir / f"{stem}.json").write_text(
                json.dumps({"segments": segs}, indent=2, ensure_ascii=False), encoding="utf-8")
            return segs, status
        last_err = RuntimeError(f"parse failed: {status}")
        time.sleep(5 * attempt)

    raise last_err or RuntimeError("All API attempts exhausted")

## 100-run serial runner

In [ ]:
_CSV_FIELDS = ["model", "student", "day", "behavior", "run", "n_segments",
               "coverage_duration_minutes", "total_duration_minutes",
               "return_code", "parse_status", "elapsed_seconds", "error"]


def run_100_serial(student, day, behavior, model_id, model_tag, start_run=1, end_run=None):
    """Run one (student, day, behavior) combination for one model, 100 times
    serially, then rebuild the summary CSV and per-run figures from disk."""
    if end_run is None:
        end_run = N_RUNS
    beh = behavior.lower()
    stu_slug = STUDENT_SLUGS[student]
    day_tag = DAY_TAGS[day]
    day_raw = DAY_RAW[day]

    # get_runs_dir() resolves to the legacy path for combinations that already
    # have complete data there (e.g. Taylor/day1/enacting) instead of assuming
    # a fresh empty directory, so idempotent skipping works for every model.
    runs_dir = get_runs_dir(stu_slug, day_raw, beh, model_tag)
    res_csv = RESULTS_ROOT / stu_slug / day_tag / beh / f"{model_tag}_100_runs.csv"
    fig_dir = FIGURES_ROOT / stu_slug / day_tag / beh
    runs_dir.mkdir(parents=True, exist_ok=True)
    fig_dir.mkdir(parents=True, exist_ok=True)

    api_csv, global_end_s = load_source_csv(student, day, beh)
    fewshot_msgs = load_prompt(beh, model_id)
    tag = f"{model_tag}_{stu_slug}_{day_tag}_{beh}"

    print(f"\n{'=' * 65}")
    print(f"RUN {start_run}-{end_run}: {student}  {day}  {beh.upper()}  [{model_id}]")
    print(f"  runs_dir     : {runs_dir}")
    print(f"  global_end_s : {global_end_s:.1f}s  ({global_end_s / 60:.2f} min)")
    print(flush=True)

    for run_idx in range(start_run, end_run + 1):
        run_dir = runs_dir / f"run{run_idx}"
        meta_p = run_dir / "run_metadata.json"

        if meta_p.exists():
            try:
                ex = json.loads(meta_p.read_text())
                succeeded = ex.get("return_code") == 0
                # A "clean" parse with no valid segments is the model genuinely
                # reporting that the behavior did not occur — a final result,
                # not a failure. It must not be silently retried: retrying would
                # spend API budget on a result that already exists, and is not
                # guaranteed to reproduce it exactly (models are not perfectly
                # deterministic even at temperature=0). Only real infrastructure
                # failures (timeouts, HTTP errors, unparseable responses) are
                # eligible for a retry.
                decided_empty = (ex.get("parse_status") == "clean"
                                  and "No valid segments" in ex.get("error", ""))
                if succeeded or decided_empty:
                    if run_idx % 10 == 0 or run_idx in (start_run, end_run):
                        tag = "already complete" if succeeded else "no behavior found (final)"
                        print(f"  [skip] run{run_idx:03d} ({tag})")
                    continue
            except Exception:
                pass

        run_dir.mkdir(parents=True, exist_ok=True)
        t0 = time.perf_counter()
        rc = 1
        err = ""
        metrics = {}
        parse_status = "failed"
        print(f"  [run {run_idx:03d}/{end_run}] {datetime.now().strftime('%H:%M:%S')}  ", end="", flush=True)

        try:
            with (run_dir / "stdout.log").open("a", encoding="utf-8") as so, \
                 (run_dir / "stderr.log").open("a", encoding="utf-8") as se:
                with redirect_stdout(so), redirect_stderr(se):
                    msgs = fewshot_msgs + [{"role": "user", "content": api_csv}]
                    segs, parse_status = call_model_once(
                        msgs, model_id, run_dir, f"{tag}_run{run_idx}", global_end_s)
                    metrics = compute_metrics(segs, global_end_s)
                    (run_dir / "segments.json").write_text(
                        json.dumps({"segments": segs}, indent=2, ensure_ascii=False), encoding="utf-8")
                    if not segs or metrics["coverage_duration_seconds"] <= 0:
                        raise ValueError(f"No valid segments (n={len(segs)})")
            rc = 0
        except Exception as exc:
            err = repr(exc)
            with (run_dir / "stderr.log").open("a", encoding="utf-8") as se:
                se.write("\n[runner exception]\n")
                se.write(traceback.format_exc())
            print(f"ERROR  {err[:80]}")
        finally:
            elapsed = round(time.perf_counter() - t0, 3)
            meta = {
                "model": model_id, "student": student, "day": day,
                "behavior": beh, "run": run_idx, "return_code": rc,
                "parse_status": parse_status, "elapsed_seconds": elapsed,
                "error": err, **metrics,
            }
            meta_p.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
            if rc == 0:
                print(f"ok  n={metrics.get('n_segments', 0):3d}  "
                      f"cov={metrics.get('coverage_duration_minutes', 0):.3f}m  "
                      f"tot={metrics.get('total_duration_minutes', 0):.3f}m  "
                      f"{elapsed:.0f}s")

    all_rows = []
    for ri in range(1, end_run + 1):
        mp = runs_dir / f"run{ri}" / "run_metadata.json"
        if mp.exists():
            try:
                all_rows.append(json.loads(mp.read_text()))
            except Exception:
                pass

    with res_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=_CSV_FIELDS, extrasaction="ignore")
        w.writeheader()
        w.writerows(all_rows)
    ok_rows = [r for r in all_rows if r.get("return_code") == 0]
    print(f"\n  Summary CSV -> {res_csv.name}  ({len(ok_rows)}/{len(all_rows)} ok)")

    xs = [r["run"] for r in all_rows]
    ns_y = [r.get("n_segments") if r.get("return_code") == 0 else None for r in all_rows]
    cov_y = [r.get("coverage_duration_minutes") if r.get("return_code") == 0 else None for r in all_rows]
    tot_y = [r.get("total_duration_minutes") if r.get("return_code") == 0 else None for r in all_rows]

    plot_series(xs, ns_y, "n_segments", f"{tag} — segment counts", fig_dir / f"{tag}_counts.png")
    plot_series(xs, cov_y, "coverage_duration_minutes", f"{tag} — coverage duration",
                fig_dir / f"{tag}_coverage_duration.png", color="C1")
    plot_series(xs, tot_y, "total_duration_minutes", f"{tag} — total duration",
                fig_dir / f"{tag}_total_duration.png", color="C2")

    print(f"  FINAL ok={len(ok_rows)}/{end_run}", flush=True)
    return all_rows

## Status table

In [ ]:
def print_status():
    print(f"{'student':<14} {'day':<6} {'behavior':<12} {'status'}")
    for student in STUDENTS:
        stu_slug = STUDENT_SLUGS[student]
        for day in DAYS:
            day_raw = DAY_RAW[day]
            for behavior in BEHAVIORS:
                beh = behavior.lower()
                key = (student, day, beh, MODEL_TAG)
                if key in EXCLUDE_ALL:
                    status = "EXCLUDED"
                elif key in SKIP_RUNNING:
                    status = "SKIP-DONE (legacy)"
                else:
                    runs_dir = get_runs_dir(stu_slug, day_raw, beh, MODEL_TAG)
                    n_ok = 0
                    if runs_dir.exists():
                        for ri in range(1, N_RUNS + 1):
                            mp = runs_dir / f"run{ri}" / "run_metadata.json"
                            if mp.exists():
                                try:
                                    if json.loads(mp.read_text()).get("return_code") == 0:
                                        n_ok += 1
                                except Exception:
                                    pass
                    status = "COMPLETE" if n_ok == N_RUNS else f"{n_ok}/{N_RUNS}"
                print(f"{student:<14} {day:<6} {behavior:<12} {status}")

## Batch runner

In [ ]:
# run_100_serial() and call_model_once() already take model_id/model_tag as
# plain parameters, so this driver works for any of the three models — it is
# not specific to Sonnet. By default it runs MODEL_ID/MODEL_TAG (Sonnet 4.6),
# because GPT-4o and GPT-5.2 already have complete, finalized results:
# re-running them would both re-spend API budget and risk producing slightly
# different output than what has already been scored and reported (the model
# calls are not perfectly deterministic even at temperature=0). Pass a
# different (model_id, model_tag) explicitly to evaluate another model.

def run_all_combos(model_id=MODEL_ID, model_tag=MODEL_TAG, only=None):
    combos = [only] if only else [(s, d, b) for s in STUDENTS for d in DAYS for b in BEHAVIORS]

    for i, (student, day, behavior) in enumerate(combos, 1):
        beh = behavior.lower()
        key = (student, day, beh, model_tag)
        print(f"\n{'#' * 70}")
        print(f"# COMBO {i}/{len(combos)}: {student} | {day} | {behavior.upper()}")
        print(f"{'#' * 70}", flush=True)

        if key not in EXCLUDE_ALL:
            if key in SKIP_RUNNING:
                print(f"  [SKIP-RUN] {model_tag} (already done at legacy path)")
            else:
                run_100_serial(student, day, behavior, model_id, model_tag)
        else:
            print(f"  [EXCLUDE] {model_tag}")

        # Gold-standard scoring has no API cost, so it is always recomputed for
        # every non-excluded model in this combination, keeping the summary
        # complete (all three models) rather than only the model just run.
        model_tags_available = [tag for _, tag in ALL_MODELS
                                 if (student, day, beh, tag) not in EXCLUDE_ALL]
        compare_gold_for_combo(student, day, behavior, model_tags_available)

    print(f"\n{'=' * 70}")
    print("BATCH COMPLETE")
    print(f"{'=' * 70}")


def run_all_models_all_combos():
    """Evaluate all three models across all 40 combinations. Safe to run in
    full at any time: runs already marked complete (return_code == 0) and
    runs where the model already gave a final "no behavior found" verdict are
    both skipped automatically (see the skip check in run_100_serial), so this
    only ever performs new work for genuinely missing or infrastructure-failed
    runs — it will not re-spend API budget on GPT-4o/GPT-5.2 results that are
    already complete and reported."""
    for model_id, model_tag in ALL_MODELS:
        print(f"\n{'*' * 70}")
        print(f"* MODEL: {model_id} ({model_tag})")
        print(f"{'*' * 70}")
        run_all_combos(model_id=model_id, model_tag=model_tag)

## Run — edit this cell, then execute

In [ ]:
# Print completion status for every combination without running anything:
# print_status()

# Evaluate a single combination / single model only:
# run_all_combos(model_id="gpt-4o", model_tag="gpt4o", only=("Taylor Swift", "day 1", "planning"))

# Evaluate all three models across all 40 combinations. Safe to run in full:
# completed runs and already-decided "no behavior found" results are both
# skipped automatically, so this only ever does new work for runs that are
# genuinely missing or failed for an infrastructure reason.
run_all_models_all_combos()